In [ ]:
import os
import shutil
import tarfile
import re

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_hub

from bs4 import BeautifulSoup

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.offline as pyo
import plotly.graph_objects as go

from wordcloud import WordCloud, STOPWORDS

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
current_folder = os.getcwd()

dataset = tf.keras.utils.get_file(
    fname="aclImdb.tar.gz",
    origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
    cache_dir=current_folder,
    extract=True
)

dataset_path = os.path.dirname(dataset)

print("Dataset path:")
print(dataset_path)

print("\nFiles:")
print(os.listdir(dataset_path))

In [ ]:
dataset_dir = os.path.join(
    dataset_path,
    "aclImdb_extracted",
    "aclImdb"
)

print("Dataset directory:")
print(dataset_dir)

print("\nContents:")
print(os.listdir(dataset_dir))

In [ ]:
train_dir = os.path.join(dataset_dir, "train")

print(os.listdir(train_dir))

In [ ]:
for file in os.listdir(train_dir):

    file_path = os.path.join(train_dir, file)

    if os.path.isfile(file_path):

        with open(file_path, "r", encoding="utf-8") as f:
            first_value = f.readline().strip()

        print(f"{file}: {first_value}")

    else:
        print(f"{file}: {file_path}")

In [ ]:
def load_dataset(directory):

    data = {
        "sentence": [],
        "sentiment": []
    }

    positive_dir = os.path.join(directory, "pos")
    negative_dir = os.path.join(directory, "neg")

    # Positive reviews
    if os.path.exists(positive_dir):

        for text_file in os.listdir(positive_dir):

            file_path = os.path.join(positive_dir, text_file)

            if os.path.isfile(file_path):

                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()

                data["sentence"].append(text)
                data["sentiment"].append(1)

    # Negative reviews
    if os.path.exists(negative_dir):

        for text_file in os.listdir(negative_dir):

            file_path = os.path.join(negative_dir, text_file)

            if os.path.isfile(file_path):

                with open(file_path, "r", encoding="utf-8") as f:
                    text = f.read()

                data["sentence"].append(text)
                data["sentiment"].append(0)

    return pd.DataFrame(data)

In [ ]:
train_dir = os.path.join(dataset_dir, "train")
test_dir = os.path.join(dataset_dir, "test")

train_df = load_dataset(train_dir)
test_df = load_dataset(test_dir)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain:")
display(train_df.head())

print("\nTest:")
display(test_df.head())

In [ ]:
sentiment_counts = train_df["sentiment"].value_counts().sort_index()

fig = px.bar(
    x=["Negative", "Positive"],
    y=sentiment_counts.values,
    color=["Negative", "Positive"],
    color_discrete_sequence=px.colors.qualitative.Dark24,
    title="Sentiments Counts"
)

fig.update_layout(
    title="Sentiments Counts",
    xaxis_title="Sentiment",
    yaxis_title="Counts",
    template="plotly_dark"
)

fig.show()

pyo.plot(
    fig,
    filename="Sentiments_Counts.html",
    auto_open=False
)

In [ ]:
def text_cleaning(text):

    # Remove HTML
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text()

    # Remove text inside square brackets
    text = re.sub(r"\[[^]]*\]", "", text)

    # Keep letters, numbers, spaces, comma and apostrophe
    text = re.sub(r"[^a-zA-Z0-9\s,']", "", text)

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
train_df["Cleaned_sentence"] = train_df["sentence"].apply(text_cleaning)

test_df["Cleaned_sentence"] = test_df["sentence"].apply(text_cleaning)

In [ ]:
print("Original:")
print(train_df["sentence"].iloc[0])

print("\nCleaned:")
print(train_df["Cleaned_sentence"].iloc[0])

In [ ]:
def generate_wordcloud(text, title):

    all_text = " ".join(text)

    wordcloud = WordCloud(
        width=800,
        height=400,
        stopwords=set(STOPWORDS),
        background_color="black"
    ).generate(all_text)

    plt.figure(figsize=(10, 5))

    plt.imshow(
        wordcloud,
        interpolation="bilinear"
    )

    plt.axis("off")
    plt.title(title)

    plt.show()

In [ ]:
positive = train_df[
    train_df["sentiment"] == 1
]["Cleaned_sentence"].tolist()

generate_wordcloud(
    positive,
    "Positive Reviews"
)

In [ ]:
negative = train_df[
    train_df["sentiment"] == 0
]["Cleaned_sentence"].tolist()

generate_wordcloud(
    negative,
    "Negative Reviews"
)

In [ ]:
Reviews = train_df["Cleaned_sentence"]
Target = train_df["sentiment"]

test_reviews = test_df["Cleaned_sentence"]
test_targets = test_df["sentiment"]

In [ ]:
x_val, x_test, y_val, y_test = train_test_split(
    test_reviews,
    test_targets,
    test_size=0.5,
    stratify=test_targets,
    random_state=42
)

In [ ]:
print("Training samples  :", len(Reviews))
print("Validation samples:", len(x_val))
print("Test samples      :", len(x_test))

In [ ]:
# Create BERT preprocessor with a limited sequence length
preprocessor = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
    "bert_base_en_uncased",
    sequence_length=128
)

# Create BERT classifier
model = keras_hub.models.BertTextClassifier.from_preset(
    "bert_base_en_uncased",
    num_classes=2,
    preprocessor=preprocessor
)

model.summary()

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["Cleaned_sentence"].values,
        train_df["sentiment"].values
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        x_val.values,
        y_val.values
    )
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (
        x_test.values,
        y_test.values
    )
)

In [ ]:
BATCH_SIZE = 8

train_dataset = train_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)

In [ ]:
optimizer = keras.optimizers.Adam(
    learning_rate=2e-5
)

model.compile(
    optimizer=optimizer,
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        )
    ]
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=3
)

In [ ]:
#Evaluate the model on the test data
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f'Test loss: {test_loss}, Test accuracy: {test_accuracy}')

In [ ]:
path = '/content'
model.save(os.path.join(path, "Model.keras"))

In [ ]:
pred_logits = loaded_model.predict(x_test.values, batch_size=8)

In [ ]:
pred_labels = np.argmax(pred_logits, axis=1)

label_map = {
    1: 'positive',
    0: 'Negative'
}

pred_labels_str = [label_map[i] for i in pred_labels]
Actual = [label_map[i] for i in y_test.values]

print('Predicted Label :', pred_labels_str[:10])
print('Actual Label    :', Actual[:10])

print("\nClassification Report:\n")
print(classification_report(Actual, pred_labels_str))

In [ ]:
def Get_sentiment(Review, Model=loaded_model):

    if not isinstance(Review, list):
        Review = [Review]

    prediction = Model.predict(Review, verbose=0)

    pred_labels = np.argmax(prediction, axis=1)
    pred_labels = [label_map[i] for i in pred_labels]

    return pred_labels

In [ ]:
Review = '''Bahubali is a blockbuster Indian movie that was released in 2015. 
It is the first part of a two-part epic saga that tells the story of a legendary hero who fights for his kingdom and his love. 
The movie has received rave reviews from critics and audiences alike for its stunning visuals, 
spectacular action scenes, and captivating storyline.'''

print(Get_sentiment(Review))